# Visual Search System — Full Pipeline

**Single notebook covering the complete pipeline (no REST API):**
- Data download + cleaning (fashion, landmarks, food)
- CLIP embedding generation
- FAISS index + visual search demo
- Image captioning + OCR enrichment *(optional — ~2 hrs)*
- Landmark detection + GeoNames + Wikidata enrichment
- Evaluation (Precision, Recall, mAP, nDCG)

**Runtime**: GPU (T4 or better) recommended. CPU works for everything except embeddings and captioning.

**Colab Secrets to add** (Key icon in sidebar):
- `KAGGLE_USERNAME`, `KAGGLE_KEY` — from kaggle.com/settings
- `GEONAMES_USERNAME` — free account at geonames.org

## 1. Mount Drive + Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
PROJECT_DIR   = '/content/drive/MyDrive/Projects/VisualSearchSystem'
LOCAL_DATA_DIR = '/content/data'

os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
print(f'Project : {PROJECT_DIR}')
print(f'Local   : {LOCAL_DATA_DIR}')

## 2. Install Dependencies

In [ ]:
!pip install -q \
    torch torchvision \
    transformers \
    faiss-cpu \
    Pillow \
    pandas numpy tqdm \
    pyyaml python-dotenv \
    kaggle \
    easyocr \
    wikipedia SPARQLWrapper \
    requests scikit-learn scipy matplotlib
print('Installation complete.')

## 3. Clone / Pull Repo

In [ ]:
import os
REPO_URL = 'https://github.com/atharvagasheTAMU/VisualSearchSystem.git'

if not os.path.exists(os.path.join(PROJECT_DIR, 'src')):
    !git clone {REPO_URL} /tmp/repo
    !cp -r /tmp/repo/. {PROJECT_DIR}/
    print('Repo cloned.')
else:
    !git -C {PROJECT_DIR} pull
    print('Repo pulled latest.')

## 4. Configure Credentials

In [ ]:
from google.colab import userdata
import os, json
from pathlib import Path

# ── Kaggle ────────────────────────────────────────────────────────────────
try:
    kaggle_dir = Path.home() / '.kaggle'
    kaggle_dir.mkdir(exist_ok=True)
    (kaggle_dir / 'kaggle.json').write_text(
        json.dumps({'username': userdata.get('KAGGLE_USERNAME'),
                    'key':      userdata.get('KAGGLE_KEY')})
    )
    (kaggle_dir / 'kaggle.json').chmod(0o600)
    print('Kaggle credentials configured.')
except Exception:
    print('Kaggle Secrets not set — add KAGGLE_USERNAME + KAGGLE_KEY to Colab Secrets.')

# ── GeoNames ──────────────────────────────────────────────────────────────
try:
    os.environ['GEONAMES_USERNAME'] = userdata.get('GEONAMES_USERNAME')
    print('GeoNames username configured.')
except Exception:
    print('GeoNames Secret not set — add GEONAMES_USERNAME for landmark enrichment.')

---
## Part 1 — Data Preparation

### 5. Download Datasets

In [ ]:
import subprocess, zipfile, shutil
from pathlib import Path

# ── Places / Landmarks dataset (OPTIONAL) ─────────────────────────────────────
#
# Pick ONE slug from the list below and paste it into PLACES_SLUG.
# All of these are public (no special terms needed).
#
#  Option A — Intel Image Classification (~350 MB, 25k images)
#             Categories: buildings, forest, glacier, mountain, sea, street
#             Slug: puneet6060/intel-image-classification
#
#  Option B — Wonders of the World (~50 MB, small but true landmarks)
#             Categories: Colosseum, Eiffel Tower, Great Wall, etc.
#             Slug: balabaskar/wonders-of-the-world-image-classification
#
#  Option C — World Famous Places (~100 MB)
#             Slug: shaistashahid/world-famous-places
#
# Leave blank to skip — pipeline works fine with just fashion + food (~145k images).

PLACES_SLUG   = 'puneet6060/intel-image-classification'   # ← change or leave blank
USE_LANDMARKS = bool(PLACES_SLUG.strip())

def _is_valid_zip(path: Path) -> bool:
    try:
        with zipfile.ZipFile(path, 'r') as zf:
            return zf.testzip() is None
    except Exception:
        return False

def _download_and_extract(ds):
    local_dir = ds['local_dir']
    drive_dir = ds['drive_dir']
    local_dir.mkdir(parents=True, exist_ok=True)
    drive_dir.mkdir(parents=True, exist_ok=True)

    # Already extracted?
    if list(local_dir.glob(ds['check_glob'])):
        print(f"  [{ds['name']}] Already on local disk."); return

    # Good zip on Drive?
    drive_zips = [z for z in drive_dir.glob('*.zip') if _is_valid_zip(z)]
    if drive_zips:
        print(f"  [{ds['name']}] Extracting from Drive zip...")
        with zipfile.ZipFile(drive_zips[0], 'r') as zf:
            zf.extractall(local_dir)
        print(f"  [{ds['name']}] Done."); return

    # Download from Kaggle
    print(f"  [{ds['name']}] Downloading {ds['size_hint']} ({ds['kaggle_slug']})...")
    result = subprocess.run(
        ['kaggle', 'datasets', 'download', '-d', ds['kaggle_slug'], '-p', str(local_dir)],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"  [{ds['name']}] DOWNLOAD FAILED:")
        print(f"    {result.stderr.strip()}")
        print(f"    → Check the slug at kaggle.com/datasets or verify credentials.")
        return

    # Find and validate zip
    local_zips = list(local_dir.glob('*.zip'))
    if not local_zips:
        print(f"  [{ds['name']}] No zip found after download."); return

    zip_path = local_zips[0]
    if not _is_valid_zip(zip_path):
        print(f"  [{ds['name']}] Zip is corrupt — deleting and re-run this cell.")
        zip_path.unlink()
        return

    # Back up to Drive, then extract
    shutil.copy(zip_path, drive_dir)
    print(f"  [{ds['name']}] Extracting...")
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(local_dir)
    print(f"  [{ds['name']}] Done.")

for ds in DATASETS:
    _download_and_extract(ds)

print('\nSummary:')
for ds in DATASETS:
    count = len(list(ds['local_dir'].glob(ds['check_glob'])))
    status = f'{count:,} images' if count else 'NOT READY'
    print(f"  {ds['name']:12} {status}")

In [ ]:
DATASETS = [
    {'name': 'fashion',
     'kaggle_slug': 'paramaggarwal/fashion-product-images-small',
     'local_dir':  Path(LOCAL_DATA_DIR) / 'raw' / 'fashion',
     'drive_dir':  Path(PROJECT_DIR)    / 'data' / 'raw' / 'fashion',
     'check_glob': 'images/*.jpg', 'size_hint': '~570 MB'},
    {'name': 'food',
     'kaggle_slug': 'dansbecker/food-101',
     'local_dir':  Path(LOCAL_DATA_DIR) / 'raw' / 'food',
     'drive_dir':  Path(PROJECT_DIR)    / 'data' / 'raw' / 'food',
     'check_glob': '**/*.jpg', 'size_hint': '~500 MB'},
]

# Add places/landmarks dataset if a slug was provided
if USE_LANDMARKS:
    DATASETS.insert(1, {
        'name': 'landmarks',
        'kaggle_slug': PLACES_SLUG,
        'local_dir':  Path(LOCAL_DATA_DIR) / 'raw' / 'landmarks',
        'drive_dir':  Path(PROJECT_DIR)    / 'data' / 'raw' / 'landmarks',
        'check_glob': '**/*.jpg', 'size_hint': 'varies',
    })
    print(f'Places/landmarks: {PLACES_SLUG}')
else:
    print('Places/landmarks skipped — using fashion + food only.')
    print('Set PLACES_SLUG in the cell above to add it.')

def _is_valid_zip(path: Path) -> bool:
    try:
        with zipfile.ZipFile(path, 'r') as zf:
            return zf.testzip() is None
    except Exception:
        return False

def _download_and_extract(ds):
    local_dir = ds['local_dir']
    drive_dir = ds['drive_dir']
    local_dir.mkdir(parents=True, exist_ok=True)
    drive_dir.mkdir(parents=True, exist_ok=True)

    # Already extracted?
    if list(local_dir.glob(ds['check_glob']))[:1]:
        print(f"  [{ds['name']}] Already on local disk."); return

    # Good zip on Drive?
    drive_zips = [z for z in drive_dir.glob('*.zip') if _is_valid_zip(z)]
    if drive_zips:
        print(f"  [{ds['name']}] Extracting from Drive zip...")
        with zipfile.ZipFile(drive_zips[0], 'r') as zf:
            zf.extractall(local_dir)
        print(f"  [{ds['name']}] Done."); return

    # Download from Kaggle
    print(f"  [{ds['name']}] Downloading {ds['size_hint']} ({ds['kaggle_slug']})...")
    result = subprocess.run(
        ['kaggle', 'datasets', 'download', '-d', ds['kaggle_slug'], '-p', str(local_dir)],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"  [{ds['name']}] DOWNLOAD FAILED:")
        print(f"    stdout: {result.stdout.strip()}")
        print(f"    stderr: {result.stderr.strip()}")
        print(f"    → Update LANDMARK_SLUG in the cell above and re-run.")
        return

    local_zips = list(local_dir.glob('*.zip'))
    if not local_zips:
        print(f"  [{ds['name']}] No zip found after download."); return

    zip_path = local_zips[0]
    if not _is_valid_zip(zip_path):
        print(f"  [{ds['name']}] Zip is corrupt — deleting, re-run to retry.")
        zip_path.unlink(); return

    shutil.copy(zip_path, drive_dir)
    print(f"  [{ds['name']}] Extracting...")
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(local_dir)
    print(f"  [{ds['name']}] Done.")

for ds in DATASETS:
    _download_and_extract(ds)

print('\nSummary:')
for ds in DATASETS:
    # Limit glob scan for speed
    imgs = list(ds['local_dir'].glob(ds['check_glob']))[:100_000]
    print(f"  {ds['name']:12} {len(imgs):>7,}+ images" if len(imgs) == 100_000
          else f"  {ds['name']:12} {len(imgs):>7,} images  {'✓' if imgs else '✗ NOT READY'}")

### 6. Load Config + Set All Paths

In [ ]:
import yaml
from pathlib import Path

with open(f'{PROJECT_DIR}/configs/config.yaml') as f:
    config = yaml.safe_load(f)

BASE  = Path(PROJECT_DIR)
LOCAL = Path(LOCAL_DATA_DIR)

# Per-dataset raw dirs on local disk (fast I/O)
for ds in config['datasets']:
    ds['raw_dir'] = str(LOCAL / 'raw' / ds['name'])

# All persistent outputs on Drive
config['paths']['metadata_csv']    = str(BASE / 'data/metadata/images.csv')
config['paths']['metadata_dir']    = str(BASE / 'data/metadata')
config['paths']['embeddings_dir']  = str(BASE / 'data/embeddings')
config['paths']['embeddings_path'] = str(BASE / 'data/embeddings/embeddings.npy')
config['paths']['id_map_path']     = str(BASE / 'data/embeddings/id_map.json')
config['paths']['index_path']      = str(BASE / 'data/embeddings/faiss.index')
config['paths']['eval_dir']        = str(BASE / 'data/eval')
config['paths']['results_dir']     = str(BASE / 'results')
config['model']['device']          = 'auto'
config['model']['batch_size']      = 128
config['evaluation']['k_values']   = [5, 10, 20]

Path(config['paths']['results_dir']).mkdir(parents=True, exist_ok=True)
print('Config ready. Paths:')
for k, v in config['paths'].items():
    print(f'  {k}: {v}')

### 7. Clean All Datasets → images.csv

In [ ]:
import sys, importlib
sys.path.insert(0, PROJECT_DIR)

# Always reload clean.py so any fixes take effect without restarting runtime
import src.preprocessing.clean as _clean_mod
importlib.reload(_clean_mod)
from src.preprocessing.clean import clean_all_datasets

from pathlib import Path
csv_path = Path(config['paths']['metadata_csv'])

if csv_path.exists():
    import pandas as pd
    existing = pd.read_csv(csv_path)
    print(f'images.csv already exists: {len(existing):,} rows')
    if 'dataset' in existing.columns:
        counts = existing.groupby('dataset').size()
        print(counts.to_string())
        # If any configured dataset is missing from the csv, force re-clean
        configured = {ds['name'] for ds in config['datasets']}
        present    = set(counts.index)
        missing    = configured - present
        if missing:
            print(f'\nDatasets missing from csv: {missing} — re-cleaning...')
            csv_path.unlink()
    if csv_path.exists():
        print('To re-clean, delete images.csv on Drive and re-run this cell.')
        df = existing

if not csv_path.exists():
    df = clean_all_datasets(config)

print(f'\nReady: {len(df):,} images')

### 8. Inspect Dataset

In [ ]:
import pandas as pd

print(f'Total images: {len(df):,}')
print(f'Columns:      {list(df.columns)}')

if 'dataset' in df.columns:
    print('\nPer dataset:')
    for name, grp in df.groupby('dataset'):
        print(f'  {name:12} {len(grp):>7,} images  |  '
              f'{grp["category"].nunique()} categories  |  '
              f'{grp["article_type"].nunique() if "article_type" in grp else "?"} article types')

print('\nTop article types:')
print(df['article_type'].value_counts().head(15).to_string())

---
## Part 2 — CLIP Embeddings

### 9. Verify GPU

In [ ]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    config['model']['device'] = 'cuda'
else:
    print('No GPU — embedding generation will be very slow on CPU.')
    print('Go to Runtime > Change runtime type > GPU')

### 10. Generate Embeddings

In [ ]:
import numpy as np, json
from pathlib import Path

emb_path    = Path(config['paths']['embeddings_path'])
id_map_path = Path(config['paths']['id_map_path'])

if emb_path.exists() and id_map_path.exists():
    emb_check = np.load(emb_path)
    with open(id_map_path) as f:
        id_map_check = json.load(f)
    print(f'Embeddings already exist: shape={emb_check.shape}, id_map={len(id_map_check):,} entries')
    REBUILD = False
else:
    print('No embeddings found — will build from scratch.')
    REBUILD = True

# Set REBUILD = True to force rebuild:
# REBUILD = True

In [ ]:
if REBUILD:
    from scripts.build_embeddings import build_embeddings
    build_embeddings(config)
else:
    print('Skipping — embeddings exist. Set REBUILD = True above to regenerate.')

# Verify
emb = np.load(config['paths']['embeddings_path'])
print(f'Embeddings shape: {emb.shape}  |  dtype: {emb.dtype}')
print(f'Sample norms (should ~1.0): {np.linalg.norm(emb[:5], axis=1).tolist()}')

---
## Part 3 — FAISS Index + Visual Search

### 11. Build FAISS Index

In [ ]:
from pathlib import Path

index_path = Path(config['paths']['index_path'])
if index_path.exists():
    print(f'Index already exists: {index_path}')
else:
    from scripts.build_index import build_index
    build_index(config)
    print('Index built.')

### 12. Initialize Searcher + Reranker

In [ ]:
from src.retrieval.search import Searcher
from src.reranking.metadata_reranker import MetadataReranker

searcher          = Searcher(config)
metadata_reranker = MetadataReranker(config)

print(f'Searcher ready.')
print(f'  Index size: {searcher.index.index.ntotal:,} vectors')
print(f'  Metadata:   {len(searcher.metadata):,} rows')

### 13. Visual Search Demo

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

def show_results(query_path, results, cols=4, title='Visual Search Results'):
    n = len(results) + 1
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3.5))
    axes = axes.flatten()
    axes[0].imshow(Image.open(query_path).resize((224, 224)))
    axes[0].set_title('QUERY', color='royalblue', fontweight='bold', fontsize=9)
    axes[0].axis('off')
    for i, r in enumerate(results):
        ax = axes[i + 1]
        img_path = r.get('path')
        if img_path and Path(img_path).exists():
            ax.imshow(Image.open(img_path).resize((224, 224)))
        else:
            ax.text(0.5, 0.5, f'ID:{r["image_id"]}', ha='center', va='center', transform=ax.transAxes)
        label = f'#{i+1} {str(r.get("article_type",""))[:14]}\n{str(r.get("dataset",""))[:8]}  {r["score"]:.3f}'
        ax.set_title(label, fontsize=7); ax.axis('off')
    for ax in axes[n:]: ax.axis('off')
    plt.suptitle(title, fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()

# Random query
import pandas as pd
df = pd.read_csv(config['paths']['metadata_csv'])
sample = df.sample(1).iloc[0]
QUERY_PATH = sample['path']
print(f'Query: ID={sample["image_id"]}  category={sample.get("category")}  '
      f'article_type={sample.get("article_type")}  dataset={sample.get("dataset")}')

results = searcher.search(QUERY_PATH, k=12)
show_results(QUERY_PATH, results)

In [ ]:
# Reranked results
meta_index = df.set_index('image_id')
qid = int(sample['image_id'])
query_meta = meta_index.loc[qid].to_dict() if qid in meta_index.index else {}

raw = searcher.search(QUERY_PATH, k=config['retrieval']['candidate_pool'])
reranked = metadata_reranker.rerank(raw, query_meta=query_meta, top_k=12)
show_results(QUERY_PATH, reranked, title='Metadata Reranked Results')

In [ ]:
# Try 3 more random queries
for i in range(3):
    s = df.sample(1).iloc[0]
    res = searcher.search(s['path'], k=8)
    print(f'\nQuery {i+1}: {s.get("article_type","")} | {s.get("dataset","")}')
    show_results(s['path'], res, cols=4)

---
## Part 4 — Image Captioning + OCR

> **Optional** — BLIP captioning takes ~2-3 hrs for ~160k images on T4.  
> Progress is saved every 200 images, so it is safe to interrupt and resume.

### 14. Caption Status Check

In [ ]:
import pandas as pd
df = pd.read_csv(config['paths']['metadata_csv'])

if 'caption' in df.columns:
    done = df['caption'].notna() & (df['caption'] != '')
    print(f'Already captioned: {done.sum():,} / {len(df):,} ({done.mean()*100:.1f}%)')
    print(f'Remaining:         {(~done).sum():,}')
else:
    print(f'No captions yet. Will caption all {len(df):,} images.')

config['model']['device']     = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'
config['model']['blip_model'] = 'Salesforce/blip-image-captioning-base'

In [ ]:
# Test BLIP on 5 samples before running full batch
from src.models.captioner import ImageCaptioner
from src.models.ocr import OCRExtractor
from PIL import Image
from IPython.display import display

captioner = ImageCaptioner(model_name=config['model']['blip_model'], device=config['model']['device'])
ocr       = OCRExtractor(gpu=config['model']['device'] == 'cuda')

for _, row in df.sample(5).iterrows():
    caption = captioner.caption_image(row['path'])
    ocr_text = ocr.extract_text(row['path'])
    print(f'[{row.get("dataset","")}] {row.get("article_type","")}')
    print(f'  Caption: {caption}')
    if ocr_text: print(f'  OCR:     {ocr_text}')
    display(Image.open(row['path']).resize((150, 150)))
    print()

In [ ]:
# Run full batch captioning (resume-safe)
from scripts.enrich_captions import enrich_captions
enrich_captions(config, batch_size=16, save_every=200)
df = pd.read_csv(config['paths']['metadata_csv'])  # reload with new columns

---
## Part 5 — Landmark Detection + GeoNames + Wikidata Enrichment

### 15. Test Landmark Detector on Sample Images

In [ ]:
from src.models.clip_encoder import CLIPEncoder
from src.models.landmark_detector import LandmarkDetector
from PIL import Image
from IPython.display import display
import pandas as pd

df = pd.read_csv(config['paths']['metadata_csv'])
encoder  = CLIPEncoder(model_name=config['model']['clip_model'], device=config['model']['device'])
detector = LandmarkDetector(encoder)

# Test on landmark images if available, otherwise random
sample_pool = df[df['dataset'] == 'landmarks'] if 'dataset' in df.columns and 'landmarks' in df['dataset'].values else df
for _, row in sample_pool.sample(min(5, len(sample_pool))).iterrows():
    result = detector.analyze(row['path'])
    print(f'[{row.get("dataset","")}] article_type={row.get("article_type","")}')
    print(f'  Detected landmark: {result["landmark"]} (conf={result["landmark_confidence"]:.3f})')
    print(f'  Scene tags:        {result["scene_tags"][:4]}')
    display(Image.open(row['path']).resize((150, 150)))
    print()

### 16. Test GeoNames + Wikidata

In [ ]:
from src.enrichment.geonames import GeoNamesResolver
from src.enrichment.wikidata import enrich_entity
from src.enrichment.wikipedia import enrich_entity_wikipedia

test_places = ['Eiffel Tower', 'Taj Mahal', 'Golden Gate Bridge', 'Colosseum']

try:
    resolver = GeoNamesResolver()
    print('GeoNames:')
    for place in test_places:
        result = resolver.resolve_place(place)
        print(f'  {place:25} -> {result}')
except EnvironmentError as e:
    print(f'GeoNames not configured: {e}')

print('\nWikidata + Wikipedia:')
for entity in test_places[:2]:
    wd = enrich_entity(entity)
    wp = enrich_entity_wikipedia(entity)
    print(f'  {entity}: type={wd.get("entity_type")} tags={wd.get("entity_tags",[])[:3]}')
    print(f'    {str(wp.get("entity_description",""))[:100]}...')

### 17. Run Full Enrichment

In [ ]:
# Location enrichment (landmark detection + GeoNames resolution)
from scripts.enrich_locations import enrich_locations
enrich_locations(config, save_every=100)

# Entity enrichment (Wikidata + Wikipedia for detected landmarks)
from scripts.enrich_entities import enrich_entities
enrich_entities(config, save_every=50)

df = pd.read_csv(config['paths']['metadata_csv'])  # reload
print(f'\nEnrichment complete. Columns: {list(df.columns)}')

In [ ]:
# Inspect enrichment results
import pandas as pd
df = pd.read_csv(config['paths']['metadata_csv'])

if 'landmark' in df.columns:
    print('Top detected landmarks:')
    print(df['landmark'].value_counts().head(10).to_string())

if 'city' in df.columns:
    has_city = df['city'].notna() & (df['city'] != '')
    print(f'\nImages with city: {has_city.sum():,}')
    if has_city.any():
        print(df[has_city][['image_id','dataset','article_type','landmark','city','country']].head(10).to_string())

---
## Part 6 — Evaluation

### 18. Build Evaluation Set

In [ ]:
import os
from pathlib import Path

eval_path = Path(config['paths']['eval_dir']) / 'eval_queries.csv'

# Rebuild if old category-level eval set exists
if eval_path.exists():
    import pandas as pd
    old = pd.read_csv(eval_path)
    if 'article_type' not in old.columns:
        print('Rebuilding eval set with article_type relevance...')
        os.remove(eval_path)

if not eval_path.exists():
    from src.evaluation.eval_set import build_eval_set
    build_eval_set(config, n_queries=200)

from src.evaluation.eval_set import load_eval_set
eval_queries = load_eval_set(config)
print(f'Eval set: {len(eval_queries)} queries')
print(f'Avg relevant per query: {sum(len(q["relevant"]) for q in eval_queries)/len(eval_queries):.1f}')

### 19. Run Evaluation

In [ ]:
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm

df = pd.read_csv(config['paths']['metadata_csv'])
df['image_id'] = df['image_id'].astype(int)
meta_index = df.set_index('image_id')

k_values       = config['evaluation']['k_values']
candidate_pool = config['retrieval']['candidate_pool']

visual_evals, reranked_evals = [], []
skipped = 0

for q in tqdm(eval_queries, desc='Evaluating'):
    qid     = q['query_image_id']
    relevant = q['relevant']
    if qid not in meta_index.index: skipped += 1; continue
    query_path = meta_index.loc[qid].get('path')
    if not query_path or not Path(query_path).exists(): skipped += 1; continue
    try:
        raw = searcher.search(query_path, k=candidate_pool)
    except Exception: skipped += 1; continue

    visual_ids = [r['image_id'] for r in raw[:max(k_values)]]
    visual_evals.append({'retrieved': visual_ids, 'relevant': relevant})

    qmeta    = meta_index.loc[qid].to_dict()
    reranked = metadata_reranker.rerank(raw, query_meta=qmeta, top_k=max(k_values))
    reranked_evals.append({'retrieved': [r['image_id'] for r in reranked], 'relevant': relevant})

print(f'Evaluated: {len(visual_evals)}  |  Skipped: {skipped}')

In [ ]:
from src.evaluation.metrics import compute_metrics, print_metrics_table

visual_metrics   = compute_metrics(visual_evals,   k_values=k_values)
reranked_metrics = compute_metrics(reranked_evals, k_values=k_values)

print_metrics_table(visual_metrics,   label='Visual Only')
print_metrics_table(reranked_metrics, label='Metadata Reranked')

improvement = reranked_metrics['mAP'] - visual_metrics['mAP']
print(f'\nmAP improvement from reranking: {improvement:+.4f} ({improvement/max(visual_metrics["mAP"],1e-6)*100:+.1f}%)')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np, csv
from pathlib import Path

metrics_to_plot = ['Recall@10', 'Precision@10', 'mAP', 'nDCG@10']
x = np.arange(len(metrics_to_plot))
w = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
b1 = ax.bar(x - w/2, [visual_metrics.get(m, 0) for m in metrics_to_plot],   w, label='Visual Only',        color='steelblue')
b2 = ax.bar(x + w/2, [reranked_metrics.get(m, 0) for m in metrics_to_plot], w, label='Metadata Reranked', color='coral')
for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(metrics_to_plot)
ax.set_ylim(0, max(max(visual_metrics.values()), max(reranked_metrics.values())) * 1.25)
ax.set_ylabel('Score'); ax.set_title('Retrieval Evaluation: Visual vs Metadata Reranked')
ax.legend()
plt.tight_layout()
chart_path = Path(config['paths']['results_dir']) / 'metrics_comparison.png'
plt.savefig(chart_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Chart saved to {chart_path}')

# Save ablation CSV
ablation_path = Path(config['paths']['results_dir']) / 'ablation.csv'
fieldnames = ['mode'] + sorted(visual_metrics.keys())
with open(ablation_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerow({'mode': 'visual_only', **visual_metrics})
    writer.writerow({'mode': 'metadata_reranked', **reranked_metrics})
print(f'Ablation CSV saved to {ablation_path}')

---
## Final Checkpoint

In [ ]:
import pandas as pd
from pathlib import Path

df       = pd.read_csv(config['paths']['metadata_csv'])
emb      = __import__('numpy').load(config['paths']['embeddings_path'])
idx_size = searcher.index.index.ntotal

assert len(df) > 0,     'images.csv is empty'
assert emb.shape[1] == 512, f'Bad embedding shape: {emb.shape}'
assert idx_size > 0,    'FAISS index is empty'
assert 'mAP' in visual_metrics, 'Evaluation not run'

print('=' * 50)
print('FULL PIPELINE COMPLETE')
print('=' * 50)
print(f'  Dataset:          {len(df):,} images')
if 'dataset' in df.columns:
    for name, grp in df.groupby('dataset'):
        print(f'    {name:12} {len(grp):,}')
print(f'  Embeddings:       {emb.shape}')
print(f'  FAISS index:      {idx_size:,} vectors')
print(f'  Visual mAP:       {visual_metrics["mAP"]:.4f}')
print(f'  Reranked mAP:     {reranked_metrics["mAP"]:.4f}')
print(f'  mAP improvement:  {reranked_metrics["mAP"]-visual_metrics["mAP"]:+.4f}')
if 'caption' in df.columns:
    captioned = (df['caption'].notna() & (df['caption'] != '')).sum()
    print(f'  Captioned:        {captioned:,} / {len(df):,}')
if 'landmark' in df.columns:
    with_landmark = df['landmark'].notna().sum()
    print(f'  With landmark:    {with_landmark:,}')